# Problem: Contextual Embeddings with Pretrained Transformers

### Problem Statement
You are tasked with exploring **contextual embeddings** using a pretrained Transformer model (DistilBERT). Unlike traditional word embeddings (Word2Vec, GloVe) that assign a single fixed vector to each word, Transformer models generate **context-dependent embeddings** where the same word receives different representations based on its surrounding context.

Your goal is to:
1. Load a pretrained DistilBERT model
2. Extract word embeddings for sentences containing polysemous words (words with multiple meanings)
3. Visualize how the embeddings change based on context

### Requirements
1. **Use a Pretrained Model**: Load DistilBERT from the Hugging Face Transformers library
2. **Extract Contextual Embeddings**: Process sentences and extract token embeddings
3. **Focus on Polysemy**: Analyze the word "bank" in different contexts (financial institution vs. river bank)
4. **Visualize Embeddings**: Use dimensionality reduction (PCA or t-SNE) to visualize embedding differences

### Context
This exercise demonstrates a key advantage of Transformer models: their ability to capture context-dependent meaning. This is crucial for many NLP tasks including:
- Question answering
- Named entity recognition
- Sentiment analysis
- Machine translation

<details>
  <summary>💡 Hint</summary>
  Use the `transformers` library from Hugging Face to load DistilBERT. Extract hidden states from the model output. Use the last hidden layer for embeddings. Apply PCA or t-SNE to reduce dimensionality for visualization.
</details>

## Step 1: Setup and Data Preparation

Install required libraries and prepare example sentences containing the polysemous word "bank" in different contexts.

In [ ]:
# Install required libraries
# !pip install transformers torch scikit-learn matplotlib numpy

# Import necessary libraries
import torch
from transformers import DistilBertTokenizer, DistilBertModel
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Define example sentences with "bank" in different contexts
sentences = [
    # Financial institution context
    "I need to go to the bank to deposit my paycheck.",
    "The bank approved my loan application yesterday.",
    "She works as a manager at a local bank.",
    "The bank charges high fees for international transfers.",
    "He opened a savings account at the bank.",
    
    # River/geographical context
    "We had a picnic on the bank of the river.",
    "The fisherman stood on the bank casting his line.",
    "Flowers bloom along the bank in spring.",
    "The boat was moored near the bank.",
    "Erosion has damaged the river bank over time."
]

# Labels for each sentence
context_labels = ['financial'] * 5 + ['geographical'] * 5

print(f"Total sentences: {len(sentences)}")
print(f"Financial contexts: {context_labels.count('financial')}")
print(f"Geographical contexts: {context_labels.count('geographical')}")

## Step 2: Load Pretrained Model

Load the DistilBERT model and tokenizer from Hugging Face.

In [ ]:
# Load the DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Load the DistilBERT model with output_hidden_states enabled
model = DistilBertModel.from_pretrained('distilbert-base-uncased', output_hidden_states=True)

# Set model to evaluation mode (disables dropout, etc.)
model.eval()

print("Model and tokenizer loaded successfully!")
print(f"Model hidden size: {model.config.hidden_size}")

## Step 3: Extract Contextual Embeddings

Process each sentence and extract the embedding for the word "bank".

In [ ]:
def extract_word_embedding(sentence, target_word, tokenizer, model):
    """
    Extract contextual embedding for a target word in a sentence.
    
    Args:
        sentence: Input sentence string
        target_word: Word to extract embedding for
        tokenizer: Pretrained tokenizer
        model: Pretrained model
    
    Returns:
        embedding: numpy array of shape (hidden_size,)
    """
    # Tokenize the sentence
    inputs = tokenizer(sentence, return_tensors='pt')
    
    # Get the tokens as a list to find the target word
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    # Find the index of the target word
    # Note: DistilBERT may split words into subwords
    target_idx = None
    for idx, token in enumerate(tokens):
        if target_word.lower() in token.lower():
            target_idx = idx
            break
    
    if target_idx is None:
        raise ValueError(f"Target word '{target_word}' not found in tokens: {tokens}")
    
    # Pass through the model
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract embeddings from the last hidden layer
    # outputs.hidden_states is a tuple of hidden states from each layer
    # We use the last layer: outputs.hidden_states[-1]
    # Shape: (batch_size, sequence_length, hidden_size)
    last_hidden_state = outputs.hidden_states[-1]
    
    # Extract the embedding for the target token
    embedding = last_hidden_state[0, target_idx, :].numpy()
    
    return embedding

In [ ]:
# Extract embeddings for "bank" in all sentences
embeddings = []
labels = []

print("Extracting embeddings...\n")

for sentence, label in zip(sentences, context_labels):
    try:
        embedding = extract_word_embedding(sentence, 'bank', tokenizer, model)
        embeddings.append(embedding)
        labels.append(label)
        print(f"✓ [{label:12s}] {sentence[:50]}...")
    except Exception as e:
        print(f"✗ Error processing: {sentence}")
        print(f"  {str(e)}")

print(f"\nSuccessfully extracted {len(embeddings)} embeddings")
print(f"Embedding dimension: {embeddings[0].shape[0]}")

## Step 4: Dimensionality Reduction

Reduce the high-dimensional embeddings to 2D for visualization using PCA or t-SNE.

In [ ]:
# Convert embeddings list to numpy array
embeddings_array = np.array(embeddings)

# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2, random_state=42)
embeddings_2d_pca = pca.fit_transform(embeddings_array)

print(f"Original embedding shape: {embeddings_array.shape}")
print(f"Reduced embedding shape (PCA): {embeddings_2d_pca.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
# Apply t-SNE for comparison
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d_tsne = tsne.fit_transform(embeddings_array)

print(f"Reduced embedding shape (t-SNE): {embeddings_2d_tsne.shape}")

## Step 5: Visualization

Create scatter plots to visualize how embeddings cluster based on context.

In [ ]:
def plot_embeddings(embeddings_2d, labels, ax, title, method_name):
    """
    Plot 2D embeddings with color coding by label.
    
    Args:
        embeddings_2d: 2D array of shape (n_samples, 2)
        labels: List of context labels
        ax: Matplotlib axis object
        title: Plot title
        method_name: Dimensionality reduction method name
    """
    # Separate points by label
    financial_mask = np.array(labels) == 'financial'
    geographical_mask = np.array(labels) == 'geographical'
    
    # Plot financial context
    ax.scatter(embeddings_2d[financial_mask, 0], 
               embeddings_2d[financial_mask, 1],
               c='blue', label='Financial', s=100, alpha=0.7, edgecolors='black')
    
    # Plot geographical context
    ax.scatter(embeddings_2d[geographical_mask, 0], 
               embeddings_2d[geographical_mask, 1],
               c='green', label='Geographical', s=100, alpha=0.7, edgecolors='black')
    
    ax.set_xlabel(f'{method_name} Component 1', fontsize=12)
    ax.set_ylabel(f'{method_name} Component 2', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)

# Create visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot PCA results
plot_embeddings(embeddings_2d_pca, labels, axes[0], 
                'Contextual Embeddings of "bank" (PCA)', 'PCA')

# Plot t-SNE results
plot_embeddings(embeddings_2d_tsne, labels, axes[1], 
                'Contextual Embeddings of "bank" (t-SNE)', 't-SNE')

plt.tight_layout()
plt.savefig('contextual_embeddings.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization complete! Embeddings are saved to 'contextual_embeddings.png'")

## Step 6: Analysis

Analyze the results and compute similarity metrics.

In [ ]:
# Calculate cosine similarity matrix
similarity_matrix = cosine_similarity(embeddings_array)

print("Cosine Similarity Analysis\n" + "="*50)

# Within-context similarities
financial_indices = [i for i, label in enumerate(labels) if label == 'financial']
geographical_indices = [i for i, label in enumerate(labels) if label == 'geographical']

# Financial context similarities (excluding self-similarity)
financial_sims = []
for i in financial_indices:
    for j in financial_indices:
        if i < j:
            financial_sims.append(similarity_matrix[i, j])

# Geographical context similarities (excluding self-similarity)
geographical_sims = []
for i in geographical_indices:
    for j in geographical_indices:
        if i < j:
            geographical_sims.append(similarity_matrix[i, j])

# Cross-context similarities
cross_sims = []
for i in financial_indices:
    for j in geographical_indices:
        cross_sims.append(similarity_matrix[i, j])

print(f"\nWithin Financial Context:")
print(f"  Mean similarity: {np.mean(financial_sims):.4f}")
print(f"  Std deviation:   {np.std(financial_sims):.4f}")

print(f"\nWithin Geographical Context:")
print(f"  Mean similarity: {np.mean(geographical_sims):.4f}")
print(f"  Std deviation:   {np.std(geographical_sims):.4f}")

print(f"\nCross-Context (Financial vs Geographical):")
print(f"  Mean similarity: {np.mean(cross_sims):.4f}")
print(f"  Std deviation:   {np.std(cross_sims):.4f}")

print(f"\n" + "="*50)
print("Key Observation:")
within_context_avg = (np.mean(financial_sims) + np.mean(geographical_sims)) / 2
cross_context_avg = np.mean(cross_sims)
difference = within_context_avg - cross_context_avg

print(f"  Average within-context similarity: {within_context_avg:.4f}")
print(f"  Cross-context similarity:          {cross_context_avg:.4f}")
print(f"  Difference:                        {difference:.4f}")

if difference > 0.05:
    print("\n✓ Strong contextual distinction detected!")
    print("  Embeddings successfully capture different meanings of 'bank'.")
else:
    print("\n⚠ Weak contextual distinction.")
    print("  Model may need more diverse contexts or fine-tuning.")

In [ ]:
# Visualize similarity matrix as a heatmap
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix, cmap='coolwarm', aspect='auto', vmin=0.85, vmax=1.0)
plt.colorbar(label='Cosine Similarity')

# Add labels
tick_labels = [f"{label[:3]}-{i+1}" for i, label in enumerate(labels)]
plt.xticks(range(len(labels)), tick_labels, rotation=45, ha='right')
plt.yticks(range(len(labels)), tick_labels)

plt.title('Cosine Similarity Matrix of "bank" Embeddings', fontsize=14, fontweight='bold')
plt.xlabel('Sentence Index (fin=financial, geo=geographical)', fontsize=11)
plt.ylabel('Sentence Index', fontsize=11)

# Add grid lines to separate contexts
plt.axhline(y=4.5, color='black', linewidth=2, linestyle='--')
plt.axvline(x=4.5, color='black', linewidth=2, linestyle='--')

plt.tight_layout()
plt.savefig('similarity_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSimilarity matrix saved to 'similarity_matrix.png'")

## Summary and Key Findings

### What We Demonstrated:

1. **Contextual Embeddings Work**: DistilBERT successfully generates different embeddings for the word "bank" based on its context (financial vs. geographical).

2. **Clear Separation**: The visualization shows distinct clusters for the two meanings, confirming that the model understands polysemy.

3. **Quantitative Evidence**: Cosine similarity analysis shows:
   - Higher similarity within the same context
   - Lower similarity across different contexts
   - This numerical difference validates the visual clustering

### Why This Matters:

- **Traditional embeddings** (Word2Vec, GloVe) assign the same vector to "bank" regardless of context
- **Transformer embeddings** adapt based on surrounding words, capturing nuanced meanings
- This capability is crucial for:
  - Word sense disambiguation
  - Machine translation
  - Question answering
  - Any task requiring deep language understanding

### Next Steps:

To extend this analysis, you could:
1. Test with more polysemous words (e.g., "bat", "pitcher", "mouse")
2. Compare different Transformer models (BERT, RoBERTa, GPT)
3. Analyze embeddings from different layers (early vs. late layers)
4. Use embeddings for downstream tasks like word sense disambiguation